In [0]:
# =============================================================================
# NOTEBOOK: 02b_silver_ohlc_history.py
# LAYER:    Silver
# PURPOSE:  Transform raw OHLC Bronze data into a clean, typed Silver table.
#
# SOURCE:   bronze/ohlc_raw        (Delta table, path-based)
# OUTPUT:   silver/ohlc_history    (Delta table, path-based, MERGE upsert)
#
# OHLC PAYLOAD STRUCTURE (from Bronze):
#   Each Bronze row has:
#     coin_id   : "bitcoin"  (from envelope extra_meta)
#     payload   : [[timestamp_ms, open, high, low, close], [...], ...]
#                 Up to ~90 arrays per coin (one per day for 90-day range)
#
# WHAT THIS NOTEBOOK DOES:
#   1. Read latest Bronze batch
#   2. Explode payload array → one row per [ts, open, high, low, close] array
#   3. Explode inner array → extract each of the 5 positional values
#   4. Convert Unix millisecond timestamp to TimestampType and DateType
#   5. Validate: high >= low, all prices >= 0, no null close/open
#   6. Deduplicate on coin_id + ohlc_timestamp
#   7. MERGE into silver/ohlc_history
#   8. OPTIMIZE + Z-ORDER
#   9. Write run log
#
# SILVER TABLE: silver_ohlc_history
#   Grain: one row per OHLC candle per coin per timestamp
#   Primary key: coin_id + ohlc_timestamp
#   Expected volume: ~50 coins × ~90 candles = ~4,500 rows per full load

In [0]:
%run ../connection

In [0]:
%run ./config

In [0]:
%run ./silver_utils

In [0]:
# =============================================================================
# CELL 1 — SETUP
# =============================================================================
 
# %run ./silver_config
# %run ./silver_utils
 
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, StringType, TimestampType, DateType
 
adls_name = "adlsnewhp"
init_silver_config(adls_name)
 
logger = get_logger("silver_ohlc_history")
 
logger.info("=" * 70)
logger.info("Silver: ohlc_history — START")
logger.info(f"  Run ID    : {SilverConfig.RUN_ID}")
logger.info(f"  Bronze in : {BronzePaths.OHLC}")
logger.info(f"  Silver out: {SilverPaths.OHLC_HISTORY}")
logger.info("=" * 70)
 

In [0]:
# =============================================================================
# CELL 2 — READ BRONZE (latest batch)
# =============================================================================
 
bronze_df, max_bronze_ts = read_bronze_incremental(
    spark, BronzePaths.COINS_MARKETS, WatermarkPaths.COINS_MARKETS,
    WatermarkPaths.WATERMARK_TABLE, logger
)

bronze_df.display()
 
assert_required_columns(
    bronze_df,
    required_cols=["coin_id", "payload", "pipeline_run_id"],
    table_name="ohlc_raw",
    logger=logger
)


In [0]:
bronze_df.display()

In [0]:
# =============================================================================
# CELL 3 — EXPLODE OUTER ARRAY (one row per candle array)
#
# Bronze payload structure per row:
#   payload: [[1700000000000, 65000.0, 66000.0, 64000.0, 65500.0],
#             [1700086400000, 65500.0, 67000.0, 65000.0, 66200.0], ...]
#
# After first explode:
#   coin_id: "bitcoin", candle: [1700000000000, 65000.0, 66000.0, 64000.0, 65500.0]
# =============================================================================
 
logger.info("CELL 3: Exploding payload array → one row per OHLC candle")
 
candles_df = (
    bronze_df
    .select(
        F.col("coin_id").cast(StringType()),
        F.col("pipeline_run_id").cast(StringType()),
        F.explode(F.col("payload")).alias("candle")   # Each element is a 5-element array
    )
)
 
logger.info(f"  Rows after first explode: {candles_df.count():,}")

In [0]:
candles_df.display()

In [0]:
# =============================================================================
# CELL 4 — EXTRACT POSITIONAL FIELDS FROM THE INNER ARRAY
#
# Each candle is an array of exactly 5 values (positional, no named keys):
#   candle[0] = timestamp in Unix milliseconds
#   candle[1] = open price (USD)
#   candle[2] = high price (USD)
#   candle[3] = low price (USD)
#   candle[4] = close price (USD)
#
# We use F.col("candle")[index] to extract each position.
#
# TIMESTAMP CONVERSION:
#   Unix milliseconds → seconds: divide by 1000
#   F.from_unixtime() takes seconds and returns a timestamp string
#   We then cast to TimestampType for proper Spark timestamp handling
# =============================================================================
 
logger.info("CELL 4: Extracting positional fields and converting timestamp")
 
typed_df = (
    candles_df
    .select(
        F.col("coin_id"),
 
        # Convert Unix ms → TimestampType
        # Step 1: divide by 1000 (ms → seconds)
        # Step 2: from_unixtime() converts integer seconds → timestamp string
        # Step 3: cast to TimestampType for Spark-native timestamp operations
        F.from_unixtime(
            (F.col("candle")[0] / 1000).cast("long")
        ).cast(TimestampType()).alias("ohlc_timestamp"),
 
        # Extract date portion from timestamp for easy daily filtering
        F.to_date(
            F.from_unixtime((F.col("candle")[0] / 1000).cast("long"))
        ).alias("ohlc_date"),
 
        F.col("candle")[1].cast(DoubleType()).alias("open_price"),
        F.col("candle")[2].cast(DoubleType()).alias("high_price"),
        F.col("candle")[3].cast(DoubleType()).alias("low_price"),
        F.col("candle")[4].cast(DoubleType()).alias("close_price"),
 
        F.col("pipeline_run_id"),
    )
    .withColumn("silver_processed_timestamp", get_silver_timestamp(SilverConfig.RUN_TS))
)
 
raw_count = typed_df.count()
logger.info(f"  Rows after field extraction: {raw_count:,}")
 
 

In [0]:
typed_df.display(_)

In [0]:
# =============================================================================
# CELL 5 — DATA QUALITY FILTERS
#
# OHLC-SPECIFIC RULES:
#   - coin_id must not be null (no anonymous candles)
#   - ohlc_timestamp must not be null (can't plot without a date)
#   - open_price and close_price must not be null or < 0
#   - high_price must be >= low_price (basic OHLC validity)
#     A candle where high < low is impossible in real market data — it means
#     the positions were swapped or the data was corrupted in transit.
# =============================================================================
 
logger.info("CELL 5: Applying OHLC data quality filters")
 
clean_df = (
    typed_df
    .filter(F.col("coin_id").isNotNull())
    .filter(F.col("ohlc_timestamp").isNotNull())
    .filter(F.col("ohlc_date").isNotNull())
    # Must have both open and close — these are required for any technical analysis
    .filter(
        F.col("open_price").isNotNull() &
        (F.col("open_price") >= DataQuality.MIN_OHLC_PRICE)
    )
    .filter(
        F.col("close_price").isNotNull() &
        (F.col("close_price") >= DataQuality.MIN_OHLC_PRICE)
    )
    # high must be >= low (physical constraint of price data)
    .filter(
        F.col("high_price").isNull() |
        F.col("low_price").isNull()  |
        (F.col("high_price") >= F.col("low_price"))
    )
)
 
clean_count = clean_df.count()
logger.info(f"  Rows after quality filters: {clean_count:,}")
 
validate_drop_rate(
    rows_before  = raw_count,
    rows_after   = clean_count,
    max_fraction = DataQuality.MAX_DROP_FRACTION,
    table_name   = "ohlc_history",
    logger       = logger,
)
 

In [0]:
# =============================================================================
# CELL 6 — DEDUPLICATE WITHIN BATCH
# Key: coin_id + ohlc_timestamp
# The same candle timestamp should not appear twice for the same coin.
# =============================================================================
 
logger.info("CELL 6: Deduplicating on coin_id + ohlc_timestamp")
 
dedup_df = clean_df.dropDuplicates(MergeKeys.OHLC_HISTORY)
 
dedup_count = dedup_df.count()
logger.info(f"  Rows after dedup: {dedup_count:,} (dropped {clean_count - dedup_count:,})")

In [0]:
# =============================================================================
# CELL 7 — FINAL COLUMN REORDER
# =============================================================================
 
final_df = dedup_df.select(*SilverColumns.OHLC_HISTORY)
log_schema(final_df, "ohlc_history_final", logger)
 

In [0]:
# =============================================================================
# CELL 8 — DELTA MERGE
# =============================================================================
 
logger.info("CELL 8: MERGE into silver/ohlc_history")
 
merge_stats = delta_merge(
    spark      = spark,
    new_df     = final_df,
    table_path = SilverPaths.OHLC_HISTORY,
    merge_keys = MergeKeys.OHLC_HISTORY,
    logger     = logger,
)
# Update watermark ONLY after successful MERGE.
# If MERGE failed, an exception would have already propagated — we never reach here.
update_watermark(
    spark          = spark,
    source_table   = WatermarkPaths.COINS_MARKETS,
    new_ts         = max_bronze_ts,
    watermark_path = WatermarkPaths.WATERMARK_TABLE,
    run_ts         = SilverConfig.RUN_TS,
    logger         = logger,
)
 
 

In [0]:
# =============================================================================
# CELL 9 — OPTIMIZE + Z-ORDER
# Z-ORDER BY coin_id: Gold reads window functions partitioned by coin_id.
# Adding ohlc_date helps Gold's date-range filters skip irrelevant files.
# =============================================================================
 
logger.info("CELL 9: OPTIMIZE silver/ohlc_history")
 
spark.sql(f"OPTIMIZE delta.`{SilverPaths.OHLC_HISTORY}` ZORDER BY (coin_id, ohlc_date)")
logger.info("  ✓ OPTIMIZE complete")
 

In [0]:
# =============================================================================
# CELL 10 — RUN LOG + COMPLETION
# =============================================================================
 
summary = {
    "notebook"                 : "02b_silver_ohlc_history",
    "pipeline_run_id"          : SilverConfig.RUN_ID,
    "run_timestamp_utc"        : SilverConfig.RUN_TS.isoformat(),
    "bronze_source"            : BronzePaths.OHLC,
    "silver_target"            : SilverPaths.OHLC_HISTORY,
    "rows_from_bronze"         : raw_count,
    "rows_after_quality_filter": clean_count,
    "rows_after_dedup"         : dedup_count,
    "merge_rows_before"        : merge_stats["rows_before"],
    "merge_rows_after"         : merge_stats["rows_after"],
    "merge_rows_inserted"      : merge_stats["rows_inserted"],
    "status"                   : "SUCCESS",
}
 
write_run_log(summary, LogPaths.OHLC_HISTORY, logger)
 
logger.info("=" * 70)
logger.info("Silver: ohlc_history — COMPLETE")
for k, v in summary.items():
    logger.info(f"  {k:<35}: {v}")
logger.info("=" * 70)
